# Day10 · E4 data-size ablation · Văn Thành Huy
So sánh 10% (142 ảnh) với 100% (1422 ảnh) trên setup E3, rank 16, seed 42, batch 2 × tích lũy 4.
Đối chứng 100% là kết quả E3 thực tế đã chạy 275 cập nhật. Lượt 10% cũng chạy đúng 275 cập nhật, dùng lịch LR được lập cho 300 bước như E3, không early stopping. Chỉ thay đổi tập ảnh train. Cùng số lượt ảnh xử lý nên 10% KHÔNG nhanh gấp 10; khoảng 40–50 phút dự trù.
**Giới hạn:** đối chứng được dừng bởi loss train và đã được xem test; đây là ablation bổ sung có chọn setup sau kết quả, không phải thí nghiệm preregistered. Không nói đã chạy 300. Đề chuẩn 300 bước chưa được đáp ứng cho E3/E4 này.
Kaggle: GPU T4 ×2, Internet ON, Add Input dataset road-damage. Chạy tuần tự. Xuất kết quả và checkpoint cuối.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, shutil, zipfile
REPO_URL = "https://github.com/thanhhuyvan/K4-Track-4-Day10-VanThanhHuy-2A202602763--VisionLanguageModel.git"
ROOT = Path("/kaggle/working/day10")
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
# Có thể trỏ đến repo được upload dưới dạng Kaggle Input nếu GitHub chưa có mã đề.
REPO_INPUT = None
DATA_INPUT = None  # None: tìm tự động trong /kaggle/input; hoặc nhập đường dẫn dataset
LOAD_MODEL = False  # đổi True khi đã bật GPU và sẵn sàng tải model

def run(*args):
    subprocess.run([str(x) for x in args], check=True, cwd=ROOT if ROOT.exists() else None)

if not ROOT.exists():
    if REPO_INPUT:
        shutil.copytree(REPO_INPUT, ROOT)
    else:
        run("git", "clone", REPO_URL, ROOT)
for required in ["scripts/prepare_data.py", "scripts/score.py", "requirements-colab.txt"]:
    assert (ROOT / required).is_file(), f"Repo thiếu {required}; upload repo vào Kaggle Input và đặt REPO_INPUT."
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
(ROOT / "work").mkdir(exist_ok=True)
(ROOT / "report/evidence").mkdir(parents=True, exist_ok=True)
# Ghi hash để phát hiện vô tình sửa công cụ đề.
PROTECTED = {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest()
             for folder in ["scripts", "tests"] for p in (ROOT / folder).rglob("*.py")}
print("Repo:", ROOT)


## 1. Môi trường — loại torchao cũ không dùng trong bài

In [ ]:
run(sys.executable, "-m", "pip", "uninstall", "-y", "torchao")
run(sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt", "pytest", "matplotlib")
check = subprocess.run([sys.executable, "-m", "pip", "check"], cwd=ROOT, capture_output=True, text=True)
print(check.stdout, check.stderr)
# pip check có thể báo xung đột thư viện khác trong image Kaggle; không giấu output.
run(sys.executable, "-m", "pytest", "-q")


In [ ]:
import torch, transformers, peft, accelerate, bitsandbytes
runtime = {"python": sys.version, "torch": torch.__version__,
           "transformers": transformers.__version__, "peft": peft.__version__,
           "accelerate": accelerate.__version__, "bitsandbytes": bitsandbytes.__version__,
           "cuda_available": torch.cuda.is_available()}
if torch.cuda.is_available():
    runtime["gpus"] = [{"name": torch.cuda.get_device_name(i),
                        "vram_gb": torch.cuda.get_device_properties(i).total_memory / 2**30}
                       for i in range(torch.cuda.device_count())]
print(json.dumps(runtime, indent=2))
(ROOT / "work/runtime.json").write_text(json.dumps(runtime, indent=2))


## 2. Dữ liệu và kiểm tra không trùng phiên quay

In [ ]:
from scripts.prepare_data import _find_root
base = Path(DATA_INPUT) if DATA_INPUT else Path("/kaggle/input")
try:
    source = _find_root(base)
except FileNotFoundError:
    raise RuntimeError("Chưa tìm thấy images/ và labels-YOLO/. Add Input dataset hoặc đặt DATA_INPUT đúng thư mục.")
run(sys.executable, "scripts/prepare_data.py", "--source", source)
meta = json.loads((ROOT / "data/splits/meta.json").read_text())
splits = {s: [json.loads(line) for line in (ROOT / f"data/splits/{s}.jsonl").read_text().splitlines() if line.strip()]
          for s in ["train", "val", "test"]}
sessions = {s: {r["session"] for r in rows} for s, rows in splits.items()}
assert not sessions["train"] & sessions["val"]
assert not sessions["train"] & sessions["test"]
assert not sessions["val"] & sessions["test"]
print(json.dumps(meta["counts"], indent=2))
print("dataset_version:", meta["dataset_version"], "— không trùng phiên quay")


## 3. Nhãn — ghi log box lỗi, giữ nguyên dataset và công cụ chấm

In [ ]:
import random, time, math, gc, re
from torch.utils.data import DataLoader
from PIL import Image

def target_box(obj):
    xc,yc,w,h = map(float,obj["yolo"])
    if w <= 0 or h <= 0: return None, "skip_zero_area"
    floats = [max(0.,min(1000.,v*1000)) for v in [xc-w/2,yc-h/2,xc+w/2,yc+h/2]]
    if floats[2]<=floats[0] or floats[3]<=floats[1]: return None,"skip_after_clipping"
    x1,y1,x2,y2=map(round,floats)
    adjusted=x2<=x1 or y2<=y1
    if x2<=x1: x1=min(x1,999); x2=x1+1
    if y2<=y1: y1=min(y1,999); y2=y1+1
    return {"label":obj["label"],"bbox_2d":[x1,y1,x2,y2]}, "expand_rounding_to_one_unit" if adjusted else None

def objects_to_answer(objects):
    boxes=[b for obj in objects for b,_ in [target_box(obj)] if b is not None]
    boxes.sort(key=lambda b:(b["bbox_2d"][1],b["bbox_2d"][0]))
    return json.dumps(boxes,separators=(",",":"))

audit=[]
for row in splits["train"]:
    for obj in row["objects"]:
        _,action=target_box(obj)
        if action: audit.append({"image_id":row["image_id"],"object":obj,"action":action})
(ROOT/"work/train_label_audit.json").write_text(json.dumps(audit,indent=2))
print("Label audit:",audit)
PROMPT='Locate all pothole, crack and manhole objects. Return only a JSON list of objects with label and bbox_2d=[x1,y1,x2,y2]. Use integer coordinates normalized to 0–1000 relative to image width and height. Return [] if none.'


## 4. Model gốc — chỉ lượng tử hóa LLM, xác nhận ViT và lm_head

In [ ]:
SETUP="E3"

def is_projector(name):
    return "visual.merger." in name or "visual.deepstack_merger_list." in name

from transformers import AutoConfig, AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig
from accelerate import init_empty_weights
from peft import LoraConfig, get_peft_model, PeftModel
import bitsandbytes as bnb
assert torch.cuda.is_available(), "Bật GPU trước khi tiếp tục"
processor=AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side="left"

def release_model():
    global model
    if "model" in globals(): del model
    gc.collect(); torch.cuda.empty_cache()

def load_base():
    config=AutoConfig.from_pretrained(MODEL_ID)
    with init_empty_weights(): skeleton=Qwen3VLForConditionalGeneration(config)
    skip=[name for name,module in skeleton.named_modules()
          if isinstance(module,torch.nn.Linear) and ("visual" in name.split(".") or name.split(".")[-1]=="lm_head")]
    assert skip
    del skeleton
    quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True,llm_int8_skip_modules=skip)
    base=Qwen3VLForConditionalGeneration.from_pretrained(MODEL_ID,
        quantization_config=quant,dtype=torch.float16,device_map={"":0})
    bad=[name for name,module in base.named_modules()
         if ("visual" in name.split(".") or name.split(".")[-1]=="lm_head")
         and isinstance(module,(bnb.nn.Linear4bit,bnb.nn.Linear8bitLt))]
    assert not bad,f"ViT/lm_head bị quantize: {bad[:10]}"
    return base

def setup_e4_data_size_0p1(base):
    for p in base.parameters(): p.requires_grad=False
    pattern=r".*language_model\.layers\.\d+\.(?:self_attn\.(?:q|k|v|o)_proj|mlp\.(?:gate|up|down)_proj)"
    vit=r".*visual\.blocks\.\d+\.(?:attn\.(?:qkv|proj)|mlp\.(?:linear_fc1|linear_fc2))"
    if SETUP == "E3": pattern=f"(?:{pattern}|{vit})"
    matched=[name for name,_ in base.named_modules() if re.fullmatch(pattern,name)]
    expected=348 if SETUP=="E3" else 252
    assert len(matched)==expected, f"LoRA targets: {len(matched)} thay vì {expected}"
    base.enable_input_require_grads()
    trained=get_peft_model(base,LoraConfig(r=16,lora_alpha=32,lora_dropout=.05,
        target_modules=pattern,task_type="CAUSAL_LM"))
    for name,p in trained.named_parameters():
        if is_projector(name):
            assert not isinstance(p,bnb.nn.Params4bit), f"Projector quantized: {name}"
            p.requires_grad=True
        if p.requires_grad:
            assert is_projector(name) or "lora_" in name, name
            p.data=p.data.float()
    projector_params=sum(p.numel() for name,p in trained.named_parameters() if is_projector(name))
    print("Projector trainable parameters:",projector_params)
    assert projector_params > 100_000_000, "Projector chưa đầy đủ"

    trained.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})
    trained.config.use_cache=False
    trained.print_trainable_parameters()
    return trained


## 5. Chat template, mask loss và collator

In [ ]:
def encode_example(row):
    with Image.open(ROOT/"data"/row["image_path"]) as im: image=im.convert("RGB")
    user={"role":"user","content":[{"type":"image","image":image},{"type":"text","text":PROMPT}]}
    prefix=processor.apply_chat_template([user],tokenize=True,add_generation_prompt=True,return_dict=True,return_tensors="pt")
    full=processor.apply_chat_template([user,{"role":"assistant","content":[{"type":"text","text":objects_to_answer(row["objects"])}]}],
        tokenize=True,add_generation_prompt=False,return_dict=True,return_tensors="pt")
    length=prefix["input_ids"].shape[1]
    assert torch.equal(full["input_ids"][:,:length],prefix["input_ids"])
    full["labels"]=full["input_ids"].clone(); full["labels"][:,:length]=-100
    assert (full["labels"]!=-100).any()
    return full

def collate(rows):
    examples=[encode_example(r) for r in rows]
    length=max(x["input_ids"].shape[1] for x in examples)
    result={}
    for key in ["input_ids","attention_mask","labels","mm_token_type_ids"]:
        if key not in examples[0]: continue
        pad=-100 if key=="labels" else processor.tokenizer.pad_token_id if key=="input_ids" else 0
        result[key]=torch.cat([torch.nn.functional.pad(x[key],(0,length-x[key].shape[1]),value=pad) for x in examples])
    for key in ["pixel_values","image_grid_thw"]: result[key]=torch.cat([x[key] for x in examples])
    return result
sample=encode_example(splits["train"][0])
print(processor.tokenizer.decode(sample["labels"][sample["labels"]!=-100].tolist()))
image_token=processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
count=int((sample["input_ids"]==image_token).sum())
assert count==220,f"Vision tokens {count}, cần 220"
print("Mask và 220 vision tokens: OK")


## 6. Training E3 — 300 cập nhật thành công; checkpoint mỗi 100 bước

In [ ]:
def train_e4_data_size_0p1(steps=300, early_stop=False, update_limit=275):
    global model
    release_model(); torch.manual_seed(42); torch.cuda.manual_seed_all(42); random.seed(42)
    model=setup_e4_data_size_0p1(load_base()); model.train()
    params=[p for p in model.parameters() if p.requires_grad]
    assert steps==300 and update_limit==275 and not early_stop
    rows=list(splits["train"]); random.shuffle(rows)
    rows=rows[:max(1,int(len(rows)*.1))]
    (ROOT/"work/e4_train_ids.json").write_text(json.dumps([r["image_id"] for r in rows],indent=2))
    loader=DataLoader(rows,batch_size=2,shuffle=True,collate_fn=collate,num_workers=0,
        generator=torch.Generator().manual_seed(42))
    iterator=iter(loader)
    opt=torch.optim.AdamW(params,lr=2e-4,weight_decay=0)
    scaler=torch.amp.GradScaler("cuda",init_scale=1024.)
    directory=ROOT/"work/checkpoints/e4_data_size_0p1"; directory.mkdir(parents=True,exist_ok=True)
    logpath=ROOT/"work/e4_data_size_0p1_run.json"
    history=[]; best_window=float("inf"); stale_windows=0; stopped_early=False; completed=attempts=overflows=0; warmup=max(1,math.ceil(steps*.05))
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); start=time.perf_counter()
    def info(status):
        torch.cuda.synchronize()
        return dict(name="e4_data_size_0p1",setup="E3",status=status,rank=16,train_fraction=.1,train_images=len(rows),ablation_variable="data_size",seed=42,
            batch_size=2,grad_accum=4,learning_rate=2e-4,requested_steps=update_limit,lr_schedule_steps=steps,
            completed_steps=completed,attempts=attempts,overflow_attempts=overflows,
            trainable_params=sum(p.numel() for p in params),
            peak_vram_gb=torch.cuda.max_memory_allocated()/2**30,
            train_minutes=(time.perf_counter()-start)/60,early_stop_enabled=early_stop,early_stop_metric="training_loss_mean_last_25_updates",early_stop_min_steps=100,early_stop_patience_windows=3,early_stop_min_delta=.005,dataset_version=meta["dataset_version"],history=history)
    def save_log(status):
        d=info(status);logpath.write_text(json.dumps(d,indent=2));return d
    def save(status):
        model.save_pretrained(directory/"adapter")
        projector={name:p.detach().cpu().clone() for name,p in model.named_parameters() if is_projector(name)}
        assert projector
        torch.save(projector,directory/"projector.pt")
        (directory/"run.json").write_text(json.dumps(save_log(status),indent=2))
    try:
        while completed<update_limit:
            attempts+=1; opt.zero_grad(set_to_none=True)
            factor=(completed+1)/warmup if completed<warmup else .5*(1+math.cos(math.pi*(completed-warmup)/max(1,steps-warmup)))
            for group in opt.param_groups:group["lr"]=2e-4*factor
            total=0.
            for _ in range(4):
                try:batch=next(iterator)
                except StopIteration:iterator=iter(loader);batch=next(iterator)
                batch={k:v.to("cuda:0") for k,v in batch.items()}
                with torch.autocast("cuda",dtype=torch.float16):loss=model(**batch).loss
                if not torch.isfinite(loss).item():raise RuntimeError("Loss không hữu hạn")
                total+=loss.item()/4;scaler.scale(loss/4).backward();del batch,loss
            bad=[name for name,p in model.named_parameters() if p.requires_grad and p.grad is not None and p.grad.dtype!=torch.float32]
            assert not bad,f"Gradient sai dtype: {bad[:10]}"
            assert any(p.grad is not None for p in params)
            scaler.unscale_(opt)
            finite=all(torch.isfinite(p.grad).all().item() for p in params if p.grad is not None)
            norm=None
            if finite:
                norm=float(torch.nn.utils.clip_grad_norm_(params,1.).item())
                if not math.isfinite(norm):raise RuntimeError("Gradient norm không hữu hạn")
            before=float(scaler.get_scale());scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True)
            if finite:completed+=1
            else:overflows+=1;print("Overflow:",attempts,before,"→",scaler.get_scale())
            history.append(dict(attempt=attempts,completed_steps=completed,loss=total,updated=finite,
                grad_norm=norm,scale_before=before,scale_after=float(scaler.get_scale()),lr=2e-4*factor))
            save_log("running")
            if overflows>=10:raise RuntimeError("10 lần overflow; dừng kiểm tra")
            if finite and (completed==1 or completed%5==0):print(f"E3 {completed}/{steps}: loss={total:.4f}")
            if finite and completed%100==0:save("running")
            # Heuristic on training loss, not validation F1. Record real update count.
            if finite and completed%25==0:
                good=[r["loss"] for r in history if r["updated"]]
                window=sum(good[-25:])/25
                if window < best_window-.005:
                    best_window=window; stale_windows=0
                elif completed>=100:
                    stale_windows+=1
                print("Loss window:",window,"stale windows:",stale_windows)
                if early_stop and completed>=100 and stale_windows>=3 and completed<steps:
                    stopped_early=True
                    print(f"Early stop at {completed}/{steps} successful updates")
                    break
        status="early_stopped" if stopped_early else "completed"
        save(status);return info(status)
    except Exception as exc:
        d=save_log("failed");d["error"]=str(exc);logpath.write_text(json.dumps(d,indent=2));raise
    finally:
        opt.zero_grad(set_to_none=True);del opt,scaler,iterator,loader;gc.collect();torch.cuda.empty_cache()


In [ ]:
# 275 cập nhật, lịch LR 300 bước để khớp đối chứng E3.
run_e4_data_size_0p1=train_e4_data_size_0p1(steps=300, early_stop=False, update_limit=275)
print({k:v for k,v in run_e4_data_size_0p1.items() if k!="history"})


In [ ]:
from IPython.display import FileLink, display
# Sao lưu ngay trước inference. Chỉ chứa adapter E3 + run.json, không model gốc.
checkpoint_zip=shutil.make_archive("/kaggle/working/day10_e4_data_size_0p1_checkpoint","zip",ROOT/"work/checkpoints/e4_data_size_0p1")
print(checkpoint_zip)
display(FileLink(checkpoint_zip))


## 7. Nạp lại adapter đã lưu — kiểm tra trên 5 ảnh val trước test

In [ ]:
release_model()
directory=ROOT/"work/checkpoints/e4_data_size_0p1"
training_info=json.loads((directory/"run.json").read_text())
assert training_info["status"] in ["completed", "early_stopped"] and training_info["completed_steps"]==275
assert training_info["dataset_version"]==meta["dataset_version"]
model=PeftModel.from_pretrained(load_base(),directory/"adapter",is_trainable=False)
projector_path=directory/"projector.pt"
assert projector_path.exists(), "Thiếu projector.pt"
saved=torch.load(projector_path,map_location="cpu",weights_only=True)
params=dict(model.named_parameters())
assert set(saved)=={name for name in params if is_projector(name)}, "Projector keys không khớp"
with torch.no_grad():
    for name,value in saved.items():
        p=params[name]
        assert p.shape==value.shape,name
        p.data=p.data.float()
        p.copy_(value.to(device=p.device,dtype=p.dtype))
del saved,params
print("Đã nạp lại adapter và toàn bộ projector")
model.eval();model.config.use_cache=True;model.gradient_checkpointing_disable()
processor.tokenizer.padding_side="left"

def predict_e4_data_size_0p1(split="val",limit=None):
    if split=="test":assert limit is None
    rows=splits[split] if limit is None else splits[split][:limit]
    path=ROOT/"work"/f"e4_data_size_0p1_{split}.jsonl"; start=time.perf_counter()
    with path.open("w",encoding="utf-8") as f:
        for i,row in enumerate(rows,1):
            with Image.open(ROOT/"data"/row["image_path"]) as im:image=im.convert("RGB")
            messages=[{"role":"user","content":[{"type":"image","image":image},{"type":"text","text":PROMPT}]}]
            inputs=processor.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,
                return_dict=True,return_tensors="pt").to("cuda:0")
            with torch.inference_mode(),torch.autocast("cuda",dtype=torch.float16):
                generated=model.generate(**inputs,max_new_tokens=256,do_sample=False)
            raw=processor.batch_decode(generated[:,inputs["input_ids"].shape[1]:],skip_special_tokens=True)[0]
            f.write(json.dumps({"image_id":row["image_id"],"raw":raw},ensure_ascii=False)+"\n");f.flush()
            if i<=3:print(row["image_id"],raw)
            if i%25==0 or i==len(rows):print(f"{split}: {i}/{len(rows)}, {(time.perf_counter()-start)/60:.1f} phút")
            del inputs,generated
    return path
preview=predict_e4_data_size_0p1("val",limit=5)
from scripts.score import parse_output
preview_rows=[json.loads(line) for line in preview.read_text().splitlines()]
print("Val parse hợp lệ:",sum(parse_output(r["raw"]) is not None for r in preview_rows),"/",len(preview_rows))


## 8. Test đầy đủ 311 ảnh và chấm E3
Chỉ chạy cell sau khi xem output val. JSON hợp lệ chưa chứng minh định vị đúng. Không sửa prompt theo kết quả test.

In [ ]:
pred=predict_e4_data_size_0p1("test")
rows=[json.loads(line) for line in pred.read_text().splitlines()]
assert len(rows)==len(splits["test"])
assert {r["image_id"] for r in rows}=={r["image_id"] for r in splits["test"]}
args=[sys.executable,"scripts/score.py","--pred",str(pred),"--split","test","--name","E4_data_size_0.1","--out","report/evidence/e4_data_size_0p1.json"]
for key in ["trainable_params","peak_vram_gb","train_minutes","completed_steps","requested_steps","batch_size","grad_accum","early_stop_enabled","train_fraction","train_images","lr_schedule_steps"]:
    args.extend(["--meta",f"{key}={training_info[key]}"])
run(*args)
print(json.loads((ROOT/"report/evidence/e4_data_size_0p1.json").read_text()))


In [ ]:
# Réutilisation du score réel E3, sans nouvelle inference sur le test.
# Provenance: user-provided day10_results (6).zip / report/evidence/e3.json.
baseline = json.loads('{\n  "n_images": 311,\n  "parse_valid_rate": 1.0,\n  "missing_predictions": 0,\n  "micro": {\n    "precision": 0.4654867256637168,\n    "recall": 0.451114922813036,\n    "f1": 0.45818815331010454,\n    "tp": 263,\n    "fp": 302,\n    "fn": 320\n  },\n  "per_class": {\n    "pothole": {\n      "precision": 0.3162393162393162,\n      "recall": 0.30578512396694213,\n      "f1": 0.3109243697478991,\n      "tp": 37,\n      "fp": 80,\n      "fn": 84\n    },\n    "crack": {\n      "precision": 0.3867924528301887,\n      "recall": 0.411371237458194,\n      "f1": 0.3987034035656402,\n      "tp": 123,\n      "fp": 195,\n      "fn": 176\n    },\n    "manhole": {\n      "precision": 0.7923076923076923,\n      "recall": 0.6319018404907976,\n      "f1": 0.7030716723549488,\n      "tp": 103,\n      "fp": 27,\n      "fn": 60\n    },\n    "other": {\n      "precision": 0.0,\n      "recall": 0.0,\n      "f1": 0.0,\n      "tp": 0,\n      "fp": 0,\n      "fn": 0\n    }\n  },\n  "mean_iou": 0.6998833691483709,\n  "fp_per_image": 0.9710610932475884,\n  "f1_ci95": [\n    0.4152191894127379,\n    0.5013428827215755\n  ],\n  "score_version": 1,\n  "name": "E3",\n  "dataset_version": "d10-21e4070301",\n  "split": "test",\n  "coord": "norm1000",\n  "meta": {\n    "trainable_params": 148426752,\n    "peak_vram_gb": 7.923566818237305,\n    "train_minutes": 44.221173763066666,\n    "completed_steps": 275,\n    "requested_steps": 300,\n    "batch_size": 2,\n    "grad_accum": 4,\n    "early_stop_enabled": "True"\n  }\n}')
assert baseline["dataset_version"] == meta["dataset_version"]
assert baseline["meta"]["completed_steps"] == 275
baseline["name"]="E4_data_size_1.0"
baseline["meta"].update({"train_fraction":1.0,"train_images":1422,
    "lr_schedule_steps":300,"reused_from":"E3",
    "source_evidence_sha256":'cbbd29d27c1e92e797fed9f1d532a7bf3d888e84cf5694b0a1629adb49b67f5f'})
(ROOT/"report/evidence/e4_data_size_1p0.json").write_text(json.dumps(baseline,indent=2))
low=json.loads((ROOT/"report/evidence/e4_data_size_0p1.json").read_text())
for label,d in [("10%",low),("100% (E3 reused)",baseline)]:
    print(label,"F1",d["micro"]["f1"],"CI95",d["f1_ci95"])


## 9. Xuất kết quả về local — tải cả hai ZIP, giữ notebook có output

In [ ]:
for rel,expected in PROTECTED.items():
    assert hashlib.sha256((ROOT/rel).read_bytes()).hexdigest()==expected,f"Đã sửa công cụ đề: {rel}"
archive=Path("/kaggle/working/day10_results.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for folder in [ROOT/"report",ROOT/"work"]:
        for p in folder.rglob("*"):
            rel=p.relative_to(ROOT)
            if p.is_file() and "checkpoints" not in rel.parts and p.suffix.lower() in {".json",".jsonl",".png",".csv",".txt",".log"}:
                z.write(p,str(rel))
    z.write(ROOT/"data/splits/meta.json","data/splits/meta.json")
print("Kết quả:",archive)
display(FileLink(str(archive)))
print("Checkpoint riêng: /kaggle/working/day10_e4_data_size_0p1_checkpoint.zip")
